In [11]:
from dotenv import load_dotenv
import os
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, END
from typing import Annotated
from typing_extensions import TypedDict, Literal
from langgraph.graph.message import add_messages

load_dotenv()

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
print("✅ LLM Groq prêt")

✅ LLM Groq prêt


In [12]:
def diagnostic_agent(state: MedicalState) -> MedicalState:
    question_count = state.get("question_count", 0)
    patient_answers = state.get("patient_answers", [])
    patient_case = state.get("patient_case", "")

    # Pas encore 5 questions → poser la suivante
    if question_count < 5:
        prompt = f"""Tu es un agent médical. Tu dois poser UNE SEULE question courte au patient.
Cas initial : {patient_case}
Questions déjà posées et réponses : {patient_answers}
Question numéro {question_count + 1}/5 — pose une question pertinente et différente des précédentes.
Réponds uniquement avec la question, rien d'autre."""

        response = llm.invoke([HumanMessage(content=prompt)])
        question = response.content
        print(f"❓ Question {question_count + 1}: {question}")

        # Simuler la réponse du patient (en vrai ce sera via l'interface)
        patient_response = input(f"👤 Votre réponse : ")

        new_answers = patient_answers + [f"Q{question_count+1}: {question} → {patient_response}"]

        return {
            "question_count": question_count + 1,
            "patient_answers": new_answers,
            "next": "diagnostic_agent"  # revenir pour poser la prochaine question
        }

    # 5 questions posées → générer la synthèse
    else:
        print("🔬 Génération de la synthèse clinique...")
        prompt = f"""Tu es un agent médical. Voici le cas d'un patient :
Cas initial : {patient_case}
Réponses aux questions : {patient_answers}

Génère une synthèse clinique préliminaire PRUDENTE en 3-4 phrases.
Rappelle que ce n'est pas un diagnostic définitif."""

        synthese = llm.invoke([HumanMessage(content=prompt)]).content

        prompt_care = f"""Basé sur cette synthèse : {synthese}
Propose une recommandation intermédiaire générale (repos, hydratation, surveillance...).
Sois prudent, 2-3 phrases maximum."""

        interim = llm.invoke([HumanMessage(content=prompt_care)]).content

        print("✅ Synthèse générée")
        return {
            "diagnostic_summary": synthese,
            "interim_care": interim,
            "next": "physician_review"
        }

print("✅ DiagnosticAgent défini")

✅ DiagnosticAgent défini


In [13]:
def supervisor(state: MedicalState) -> MedicalState:
    next_node = state.get("next", "diagnostic_agent")
    print(f"🔀 Supervisor → {next_node}")
    return {"next": next_node}

def physician_review(state: MedicalState) -> MedicalState:
    print("\n📋 SYNTHÈSE POUR LE MÉDECIN :")
    print(state.get("diagnostic_summary"))
    print("\n💊 RECOMMANDATION INTERMÉDIAIRE :")
    print(state.get("interim_care"))
    treatment = input("\n👨‍⚕️ Médecin — traitement proposé : ")
    return {"physician_treatment": treatment, "next": "report_agent"}

def report_agent(state: MedicalState) -> MedicalState:
    answers_text = "\n".join(state.get("patient_answers", []))
    rapport = f"""
╔══════════════════════════════════════╗
       RAPPORT CLINIQUE PRÉLIMINAIRE
╚══════════════════════════════════════╝

📌 CAS PATIENT : {state.get('patient_case')}

📝 RÉPONSES AUX QUESTIONS :
{answers_text}

🔬 SYNTHÈSE CLINIQUE :
{state.get('diagnostic_summary')}

💊 RECOMMANDATION INTERMÉDIAIRE :
{state.get('interim_care')}

👨‍⚕️ TRAITEMENT MÉDECIN :
{state.get('physician_treatment')}

⚠️  Ce système ne remplace pas une consultation médicale.
"""
    print(rapport)
    return {"final_report": rapport, "next": "FINISH"}

print("✅ Tous les agents définis")

✅ Tous les agents définis


In [14]:
builder = StateGraph(MedicalState)
builder.add_node("supervisor", supervisor)
builder.add_node("diagnostic_agent", diagnostic_agent)
builder.add_node("physician_review", physician_review)
builder.add_node("report_agent", report_agent)
builder.set_entry_point("supervisor")

builder.add_conditional_edges(
    "supervisor",
    lambda s: s.get("next"),
    {
        "diagnostic_agent": "diagnostic_agent",
        "physician_review": "physician_review",
        "report_agent": "report_agent",
        "FINISH": END
    }
)

builder.add_edge("diagnostic_agent", "supervisor")
builder.add_edge("physician_review", "supervisor")
builder.add_edge("report_agent", "supervisor")

graph = builder.compile()
print("✅ Graphe compilé")

✅ Graphe compilé


In [15]:
result = graph.invoke({
    "patient_case": "Patient de 28 ans, maux de tête depuis 3 jours, légère fièvre.",
    "question_count": 0,
    "patient_answers": [],
    "next": "diagnostic_agent"
})

🔀 Supervisor → diagnostic_agent
❓ Question 1: Avez-vous ressenti des difficultés à vous concentrer ou à vous souvenir de choses récentes ?
🔀 Supervisor → diagnostic_agent
❓ Question 2: Avez-vous ressenti des douleurs ou des pressions dans votre visage ou votre tête qui se déplacent lorsque vous bougez ?
🔀 Supervisor → diagnostic_agent
❓ Question 3: Avez-vous eu des vomissements ou des nausées depuis le début de vos maux de tête ?
🔀 Supervisor → diagnostic_agent
❓ Question 4: Avez-vous des difficultés à voir ou à distinguer les couleurs depuis le début de vos maux de tête ?
🔀 Supervisor → diagnostic_agent
❓ Question 5: Avez-vous déjà eu des maux de tête similaires dans le passé ?
🔀 Supervisor → diagnostic_agent
🔬 Génération de la synthèse clinique...
✅ Synthèse générée
🔀 Supervisor → physician_review

📋 SYNTHÈSE POUR LE MÉDECIN :
**Synthèse clinique préliminaire**

Le patient de 28 ans présente des maux de tête persistants depuis 3 jours, associés à une légère fièvre et des vomissements